<a href="https://colab.research.google.com/github/jmontseny/espec_ADD/blob/main/Sprint7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from numpy import random
import string

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Caso 1: Selección de Personal

Trabajas como analista de datos en el departamento de RRHH de una empresa. Se ha abierto una nueva oferta laboral interna y la empresa ha decidido que sólo las 10 primeras personas que se inscriban podrán participar en el proceso de selección.

Para ello debemos desarrollar una herramienta sencilla en Python que actúe como formulario de inscripción y sistema de clasificación inicial.
Este formulario debe permitir introducir para cada candidato:
- Nombre y apellidos
- Teléfono de contacto
- Años de experiencia en la empresa
- Conocimiento de Catalán (0-3 puntos)
  - 0 = Sin conocimiento
  - 1 = Hasta nivel B1
  - 2 = Hasta nivel B2
  - 3 = C1 o superior
- Dominio de Excel (0-3 puntos)
  - 0 = Nivel básico
  - 1 = Uso de funciones
  - 2 = Tablas dinámicas
  - 3 = Visual Basic y Macros
- Título académico (0-3 puntos)
  - 0 = ESO
  - 1 = FP
  - 2 = Universidad
  - 3 = Máster o superior
Nos aseguraremos de que toda la informaciónintroducida es correcta y entra dentro de los valores permitidos. A partir de estos datos, el programa tendrá que calcular una puntuación total aplicando el peso de la experiencia (años * 1.5) y smando los puntos de catalán, excel, y titulación, para a continuación clasificar automáticamente cada persona según los límites establecidos por la empresa.
- Prioridad Alta → puntuación ≥ 12
- Candidato adecuado → 8 ≤ puntuación < 12
- No Prioritario → puntuación < 8
Cada vez que alguien complete el formulario, se ha de mostrar su puntuación individual. Cuando se hayan inscrito exactamente 10 personas, la herramienta debe dejar de aceptar nuevos registros y generar un informe final enpantalla, ordenado primer según la prioridad y, en caso de empate, por la puntuación obtenida. Este informe debe incluir el nombre, teléfono, y la puntuación de cada candidato para que el departamento de RRHH los pueda contactar rápidamente.

## Tenemos una lista con datos y a partir de ella para resolver el problema:

1. Obtendremos los datos de la lista (inputs_test) y los agruparemos en un diccionario por cada candidato
2. Chequearemos que los datos entrantes sean válidos y dentro de los rangos correctos
3. Una vez validados, calcularemos la puntuación de cada candidato
4. Con la puntuación, clasificaremos los candidatos en base a su puntuación ( ≥ 12 = Prioridad alta; 8-11 = Candidato adecuado; < 8 = No prioritario)
5. Mostramos el nombre de los candidatos válidos y su puntuación
6. Añadimos validación para no superar los 10 candidatos válidos
7. Ordenamos los candidatos en función de la prioridad y puntuación
8. Mostramos los candidatos, su información de contacto, y su puntuación

In [2]:
inputs_test = [
    "Laura Serra", "608112233", 6, 3, 3, 3,
    "Clara Gómez", "655221199", 3, 3, 3, 2,
    "Sergio López", "644112211", 3, 2, 2, 2,
    "Candidat Error", "611000000", 5, 7, 3, 2, # Catalán = 7 → NO VÁLIDO
    "Marta Ruiz", "699887766", 2, 3, 2, 1,
    "Marc Vidal", "612443322", 6, 3, 3, 3,
    "Júlia Romero", "677998877", 5, 3, 3, 2,
    "Pol Navarro", "699554433", 4, 3, 2, 2,
    "David Ferrer", "677665544", 2, 1, 1, 1,
    "Anna Paredes", "633221144", 1, 2, 1, 0,
    "Anibal Paredes", "644272727", 1, 2, 1, 0,
    "Núria Casals", "600112200", 8, 2, 3, 3 # 11r candidato
]

In [3]:
# función para obtener los datos de los candidatos y crear un diccionario donde almacenar la información

def obtener_datos(nombre, telefono, experiencia, catalan, excel, titulo):

    return {
        "nombre": nombre,
        "telefono": telefono,
        "experiencia": experiencia,
        "catalan": catalan,
        "excel": excel,
        "titulo": titulo
    }

In [4]:
# para probar las funciones, creamos una lista de prueba y obtenemos los datos, guardándolos en candidatos_prueba

candidatos_prueba = []

for i in range(0, len(inputs_test), 6):

    candidato_prueba = obtener_datos(
        inputs_test[i],
        inputs_test[i+1],
        inputs_test[i+2],
        inputs_test[i+3],
        inputs_test[i+4],
        inputs_test[i+5]
    )

    candidatos_prueba.append(candidato_prueba)

# mostramos por pantalla como queda el primer candidato

print(candidatos_prueba[0])

{'nombre': 'Laura Serra', 'telefono': '608112233', 'experiencia': 6, 'catalan': 3, 'excel': 3, 'titulo': 3}


In [5]:
# función para validar que los datos introducidos sean coherentes con cómo vamos a trabajar con ellos

def validar_datos(candidato):

    try:
        candidato['experiencia'] = int(candidato['experiencia'])
        candidato['catalan'] = int(candidato['catalan'])
        candidato['excel'] = int(candidato['excel'])
        candidato['titulo'] = int(candidato['titulo'])

        return candidato

    except ValueError:
        print("Error: Por favor, ingrese números válidos")
        return None

In [6]:
# aplicamos la validación al primer índice de candidatos_prueba y vemos el resultado

print(validar_datos(candidatos_prueba[0]))

{'nombre': 'Laura Serra', 'telefono': '608112233', 'experiencia': 6, 'catalan': 3, 'excel': 3, 'titulo': 3}


In [7]:
# función de validacion de los rangos para experiencia, catalán, excel y titulo

def validar_rangos(candidato):

    if (
        candidato['experiencia'] >= 0
        and 0 <= candidato['catalan'] <= 3
        and 0 <= candidato['excel'] <= 3
        and 0 <= candidato['titulo'] <= 3
    ):
        return True

    else:
        print("Error: Los valores de experiencia, catalán, excel, o título están fuera de los rangos válidos")
        return False

In [8]:
# aplicamos la función a los candidatos la lista de prueba

for candidato in candidatos_prueba:
  validar_rangos(candidato)

# vemos qué devuelve aplicando la función al primer candidato de la lista de prueba

print(validar_rangos(candidatos_prueba[0]))

Error: Los valores de experiencia, catalán, excel, o título están fuera de los rangos válidos
True


In [9]:
# función con la que calcular la puntuación del candidato con la formula:
# puntuación = (experiencia * 1.5) + nivel catalan + nivel excel + nivel estudios

def calcular_puntuacion(candidato):

    puntuacion = (candidato['experiencia']*1.5) + candidato['catalan'] + candidato['excel'] + candidato['titulo']

    candidato['puntuacion'] = puntuacion

    return candidato['puntuacion']

In [10]:
# aplicamos la función a los candidatos la lista de prueba

for candidato in candidatos_prueba:
  calcular_puntuacion(candidato)

# vemos qué devuelve aplicando la función al primer candidato de la lista de prueba

print(calcular_puntuacion(candidatos_prueba[0]))

18.0


In [11]:
# tomando el valor obtenido del cálculo de la puntuación, con esta función sacamos la clasificación teniendo en cuenta que:
# Prioridad Alta → puntuación ≥ 12
# Candidato Adecuado → 8 ≤ puntuación < 12
# No Prioritario → puntuació < 8

def clasificar_candidato(candidato):

    if candidato['puntuacion'] >= 12:
        clasificacion = "Prioridad Alta"

    elif 8 <= candidato['puntuacion'] < 12:
        clasificacion = "Candidato Adecuado"

    else:
        clasificacion = "No Prioritario"

    candidato['clasificacion'] = clasificacion

    return candidato['clasificacion']

In [12]:
# aplicamos la función a los candidatos la lista de prueba

for candidato in candidatos_prueba:
  clasificar_candidato(candidato)

# vemos qué devuelve aplicando la función a dos candidatos diferentes de la lista de prueba

print(clasificar_candidato(candidatos_prueba[0]))
print(clasificar_candidato(candidatos_prueba[2]))

Prioridad Alta
Candidato Adecuado


In [13]:
# función para mostrar el nombre del candidato y su puntuación

def mostrar_individuo(candidato):

    print(
        f"Nombre: {candidato['nombre']} | "
        f"Puntuación: {candidato['puntuacion']}"
    )

In [14]:
# vemos qué devuelve aplicando la función al primer candidato de la lista de prueba

print(mostrar_individuo(candidatos_prueba[0]))

Nombre: Laura Serra | Puntuación: 18.0
None


In [15]:
# creamos un diccionario de prioridades para asignar una prioridad numérica y que,
# en caso de empate de prioridad, ordenaremos descendentemente en función de la puntuación

prioridades = {
    "Prioridad Alta": 3,
    "Candidato Adecuado": 2,
    "No Prioritario": 1
}

In [16]:
# función para ordenar a los candidatos en base a la clasificación y en caso de empate por la puntuación

def ordenar_candidatos(candidatos):

    candidatos_ordenados = sorted(
        candidatos,
        key=lambda candidato: (
            prioridades[candidato['clasificacion']],
            candidato['puntuacion']
        ),
        reverse=True
    )

    return candidatos_ordenados

In [17]:
# aplicamos la función a los primeros candidatos de la lista de prueba y vemos el resultado ordenado

print(ordenar_candidatos(candidatos_prueba[:3]))

[{'nombre': 'Laura Serra', 'telefono': '608112233', 'experiencia': 6, 'catalan': 3, 'excel': 3, 'titulo': 3, 'puntuacion': 18.0, 'clasificacion': 'Prioridad Alta'}, {'nombre': 'Clara Gómez', 'telefono': '655221199', 'experiencia': 3, 'catalan': 3, 'excel': 3, 'titulo': 2, 'puntuacion': 12.5, 'clasificacion': 'Prioridad Alta'}, {'nombre': 'Sergio López', 'telefono': '644112211', 'experiencia': 3, 'catalan': 2, 'excel': 2, 'titulo': 2, 'puntuacion': 10.5, 'clasificacion': 'Candidato Adecuado'}]


In [18]:
# función para mostrar el nombre de los candidatos, teléfono, y puntuación una vez ya ordenados

def mostrar_informe(candidatos):

    for candidato in candidatos:
        print(
            f"Nombre: {candidato['nombre']} | "
            f"Teléfono: {candidato['telefono']} | "
            f"Puntuación: {candidato['puntuacion']}"
        )

In [19]:
# aplicamos la función a los 3 primeros candidatos de la lista y vemos el resultado

print(mostrar_informe(candidatos_prueba[:3]))

Nombre: Laura Serra | Teléfono: 608112233 | Puntuación: 18.0
Nombre: Clara Gómez | Teléfono: 655221199 | Puntuación: 12.5
Nombre: Sergio López | Teléfono: 644112211 | Puntuación: 10.5
None


In [20]:
# función principal

def main_seleccion(lista_candidatos):

    candidatos_final = []

    # para este ejercicio, teniendo la información de los candidatos en inputs_test, podemos extraer la información iterando.
    # la información de cada candidato se almacena en bloques de 6, por lo que cogemos los datos de 6 en 6, y saltamos cada 6,
    # separando así cada candidato.

    for i in range(0, len(lista_candidatos), 6):

        candidato = obtener_datos(
            lista_candidatos[i],
            lista_candidatos[i+1],
            lista_candidatos[i+2],
            lista_candidatos[i+3],
            lista_candidatos[i+4],
            lista_candidatos[i+5]
        )

        # validamos los rangos y guardamos el resultado
        candidato = validar_datos(candidato)

        # si la validación falla, pasamos al siguiente candidato
        if candidato is None:
            continue

        # validamos los rangos y si True aplicamos el resto de funciones
        if validar_rangos(candidato):

            calcular_puntuacion(candidato)

            clasificar_candidato(candidato)

            # si el candidato ha pasado la validación y una vez calculada la puntuación y clasificarlo, lo añadimos a la lista candidatos
            candidatos_final.append(candidato)

            mostrar_individuo(candidato)

        # añadimos una validación de cuantos candidatos válidos han entrado en la lista para que rompa el bucle
        # cuando lleguemos a 10 candidatos válidos
        if len(candidatos_final) == 10:
            break

    # añadimos la función que ordenará a los candidatos en función de su prioridad y puntuación
    candidatos_final = ordenar_candidatos(candidatos_final)

    # mostramos informe final
    print("\n--- INFORME FINAL RRHH ---")

    mostrar_informe(candidatos_final)

In [21]:
main_seleccion(inputs_test)

Nombre: Laura Serra | Puntuación: 18.0
Nombre: Clara Gómez | Puntuación: 12.5
Nombre: Sergio López | Puntuación: 10.5
Error: Los valores de experiencia, catalán, excel, o título están fuera de los rangos válidos
Nombre: Marta Ruiz | Puntuación: 9.0
Nombre: Marc Vidal | Puntuación: 18.0
Nombre: Júlia Romero | Puntuación: 15.5
Nombre: Pol Navarro | Puntuación: 13.0
Nombre: David Ferrer | Puntuación: 6.0
Nombre: Anna Paredes | Puntuación: 4.5
Nombre: Anibal Paredes | Puntuación: 4.5

--- INFORME FINAL RRHH ---
Nombre: Laura Serra | Teléfono: 608112233 | Puntuación: 18.0
Nombre: Marc Vidal | Teléfono: 612443322 | Puntuación: 18.0
Nombre: Júlia Romero | Teléfono: 677998877 | Puntuación: 15.5
Nombre: Pol Navarro | Teléfono: 699554433 | Puntuación: 13.0
Nombre: Clara Gómez | Teléfono: 655221199 | Puntuación: 12.5
Nombre: Sergio López | Teléfono: 644112211 | Puntuación: 10.5
Nombre: Marta Ruiz | Teléfono: 699887766 | Puntuación: 9.0
Nombre: David Ferrer | Teléfono: 677665544 | Puntuación: 6.0


################################################################

# Caso 2: Segmentación por riesgo de abandono de carrito de compra

Trabajas como analista de datos en el departamento de Marketing Digital. La empresa quiere mejorar la efectividad de las campañas de recuperación de carritos de compra abandonados y necesita identificar qué clientes tienen mayor riesgo de abandono, qué segmento de comportamiento presentan y qué mensaje publicitario es el más adecuado para cada uno.

Dispones de un conjunto de datos con información básica de 10 clientes:

- id_cliente
- nombre
- edad
- grupo_edad
- codigo_comportamiento

El departamento de CRM ha definido manualmente:

Códigos de comportamiento:

- 0 → Navega pero no añade productos
- 1 → Añade productos pero no llega al checkout
- 2 → Llega al checkout pero no paga
- 3 → Carrito abandonado hace menos de 24 h
- 4 → Carrito abandonado hace más de 24 h

Factores según código de comportamiento:

- Código 0 → 0.10
- Código 1 → 0.25
- Código 2 → 0.50
- Código 3 → 0.75
- Código 4 → 0.90

Factores según grupo de edad:

- 18-25 → 1.20
- 26-40 → 1.00
- 41-60 → 0.85
- 60+ → 0.70

Fórmula del riesgo final:

riesgo_final = factor_codigo * factor_edad

Una vez procesados los clientes, el equipo quiere que seamos capaces de responder a la siguientes preguntas interpretando el riesgo final y el segmento:

- ¿Qué clientes deben ser contactados primero en una campaña de retargeting?
- ¿Qué diferencias existen entre clientes con el mismo comportamiento pero edades diferentes? (Por ejemplo: ¿un cliente joven que abandona el carrito hace 24 h tiene el mismo riesgo que uno de mayor edad?)
- ¿Qué clientes muestran tan poco interés que no merece la pena invertir presupuesto en ellos?
- ¿Qué cliente sería el más rentable de recuperar?

## Para resolver el problema:

1. Crear un diccionario de diccionarios con la información de cada código de comportamiento (segmento, mensaje y factor de riesgo)
2. Crear un diccionario con los factores asociados a cada grupo de edad
3. Recorrer la lista de clientes y obtener el segmento, mensaje y factor de comportamiento según el código de cada cliente
4. Obtener el factor de edad correspondiente al grupo de edad de cada cliente
5. Calcular el riesgo final multiplicando el factor de comportamiento por el factor de edad
6. Añadir toda la información calculada al diccionario de cada cliente
7. Procesar todos los clientes mediante una función principal que coordine los pasos anteriores
8. Convertir la información resultante en un DataFrame para facilitar su análisis
9. Analizar los resultados para identificar:

- Los clientes con mayor riesgo de abandono
- Las diferencias de riesgo entre grupos de edad
- Los clientes con menor interés comercial
- Los clientes más rentables para una campaña de recuperación

10. Mostrar el DataFrame final y las respuestas a las preguntas planteadas por el departamento de Marketing

In [22]:
# diccionario de clientes

clientes_marketing = [
    {"id_cliente": 1, "nombre": "Laura Serra", "edad": 23, "grupo_edad": "18-25", "codigo_comportamiento": 4},
    {"id_cliente": 2, "nombre": "Marc Vidal", "edad": 35, "grupo_edad": "26-40", "codigo_comportamiento": 3},
    {"id_cliente": 3, "nombre": "Ana López", "edad": 29, "grupo_edad": "26-40", "codigo_comportamiento": 2},
    {"id_cliente": 4, "nombre": "Joan Riera", "edad": 19, "grupo_edad": "18-25", "codigo_comportamiento": 2},
    {"id_cliente": 5, "nombre": "Mónica Pérez", "edad": 47, "grupo_edad": "41-60", "codigo_comportamiento": 4},
    {"id_cliente": 6, "nombre": "Luis García", "edad": 52, "grupo_edad": "41-60", "codigo_comportamiento": 1},
    {"id_cliente": 7, "nombre": "Pilar Sánchez", "edad": 61, "grupo_edad": "60+", "codigo_comportamiento": 3},
    {"id_cliente": 8, "nombre": "Eva Martín", "edad": 38, "grupo_edad": "26-40", "codigo_comportamiento": 0},
    {"id_cliente": 9, "nombre": "Diego Romero", "edad": 24, "grupo_edad": "18-25", "codigo_comportamiento": 1},
    {"id_cliente": 10, "nombre": "Núria Costa", "edad": 33, "grupo_edad": "26-40", "codigo_comportamiento": 4}
]

In [23]:
# creamos los diccionarios con la información facilitada por CRM
# y añadimos también un mensaje a cada código de comportamiento

segmento_factor = {
    0: {
        "segmento": "Navega pero no añade productos",
        "factor": 0.10,
        "mensaje": "Descubre nuestros productos más populares y encuentra algo para ti."
    },

    1: {
        "segmento": "Añade productos pero no llega al checkout",
        "factor": 0.25,
        "mensaje": "Tus productos siguen esperándote. Completa tu compra en pocos clics."
    },

    2: {
        "segmento": "Llega al checkout pero no paga",
        "factor": 0.50,
        "mensaje": "Finaliza tu pedido ahora y disfruta de una experiencia de compra rápida y segura."
    },

    3: {
        "segmento": "Cesta abandonada hace menos de 24 horas",
        "factor": 0.75,
        "mensaje": "¡No olvides tu cesta! Completa tu compra antes de que se agoten los productos."
    },

    4: {
        "segmento": "Cesta abandonada hace más de 24 horas",
        "factor": 0.90,
        "mensaje": "Te echamos de menos. Recupera tu cesta ahora y aprovecha una oferta exclusiva."
    }
}

factores_edad = {
    "18-25": 1.20,
    "26-40": 1.00,
    "41-60": 0.85,
    "60+": 0.70
}

In [24]:
# función para iterar por los clientes de la lista, y añadir al diccionario de cada cliente su segmento, factor según el código, y mensaje

def obtener_segmento_y_factor(lista_clientes):

    for cliente in lista_clientes:

        cliente['segmento'] = segmento_factor[cliente['codigo_comportamiento']]['segmento']
        cliente['factor_codigo'] = segmento_factor[cliente['codigo_comportamiento']]['factor']
        cliente['mensaje'] = segmento_factor[cliente['codigo_comportamiento']]['mensaje']

    return lista_clientes

In [25]:
# creamos una lista para probar las funciones copiando clientes_marketing

clientes_marketing_prueba = clientes_marketing

# aplicamos la función a la lista de prueba clientes_marketing_prueba

clientes_marketing_prueba = obtener_segmento_y_factor(clientes_marketing_prueba)

# vemos cómo queda el primer cliente de la lista de prueba

print(clientes_marketing_prueba[0])

{'id_cliente': 1, 'nombre': 'Laura Serra', 'edad': 23, 'grupo_edad': '18-25', 'codigo_comportamiento': 4, 'segmento': 'Cesta abandonada hace más de 24 horas', 'factor_codigo': 0.9, 'mensaje': 'Te echamos de menos. Recupera tu cesta ahora y aprovecha una oferta exclusiva.'}


In [26]:
# función para iterar por los clientes de la lista y en base al grupo de edad añadir el factor de riesgo de edad

def obtener_factor_edad(lista_clientes):

    for cliente in lista_clientes:

        cliente['factor_edad'] = factores_edad[cliente['grupo_edad']]

    return lista_clientes

In [27]:
# aplicamos la función a la lista de prueba

clientes_marketing_prueba = obtener_factor_edad(clientes_marketing_prueba)

# vemos el resultado del primer cliente de la lista de prueba

print(clientes_marketing_prueba[0])

{'id_cliente': 1, 'nombre': 'Laura Serra', 'edad': 23, 'grupo_edad': '18-25', 'codigo_comportamiento': 4, 'segmento': 'Cesta abandonada hace más de 24 horas', 'factor_codigo': 0.9, 'mensaje': 'Te echamos de menos. Recupera tu cesta ahora y aprovecha una oferta exclusiva.', 'factor_edad': 1.2}


In [28]:
# función para calcular el riesgo final en base al factor por código y por edad

def calcular_riesgo(lista_clientes):

    for cliente in lista_clientes:

        cliente['riesgo_final'] = cliente['factor_codigo'] * cliente['factor_edad']

    return lista_clientes

In [29]:
# aplicamos la función a la lista de prueba

clientes_marketing_prueba = calcular_riesgo(clientes_marketing_prueba)

# vemos el resultado del primer cliente de la lista de prueba

print(clientes_marketing_prueba[0])

{'id_cliente': 1, 'nombre': 'Laura Serra', 'edad': 23, 'grupo_edad': '18-25', 'codigo_comportamiento': 4, 'segmento': 'Cesta abandonada hace más de 24 horas', 'factor_codigo': 0.9, 'mensaje': 'Te echamos de menos. Recupera tu cesta ahora y aprovecha una oferta exclusiva.', 'factor_edad': 1.2, 'riesgo_final': 1.08}


In [30]:
# función para procesar la lista y la enriquecerla

def procesar_clientes(lista_clientes):

    obtener_segmento_y_factor(lista_clientes)

    obtener_factor_edad(lista_clientes)

    calcular_riesgo(lista_clientes)

    return lista_clientes

In [31]:
# aplicamos la función a la lista de prueba

clientes_marketing_prueba = procesar_clientes(clientes_marketing_prueba)

# vemos el resultado del primer cliente de la lista de prueba

print(clientes_marketing_prueba[0])

{'id_cliente': 1, 'nombre': 'Laura Serra', 'edad': 23, 'grupo_edad': '18-25', 'codigo_comportamiento': 4, 'segmento': 'Cesta abandonada hace más de 24 horas', 'factor_codigo': 0.9, 'mensaje': 'Te echamos de menos. Recupera tu cesta ahora y aprovecha una oferta exclusiva.', 'factor_edad': 1.2, 'riesgo_final': 1.08}


In [32]:
# función para convertir la información a un DataFrame

def convertir_dataframe(lista_clientes):

    df_clientes = pd.DataFrame(lista_clientes)

    return df_clientes

In [33]:
# aplicamos la función a la lista de prueba

df_clientes_prueba = convertir_dataframe(clientes_marketing_prueba)

# vemos el resultado de los primero 3 clientes del DataFrame

print(df_clientes_prueba.head(3))

   id_cliente       nombre  edad grupo_edad  codigo_comportamiento  \
0           1  Laura Serra    23      18-25                      4   
1           2   Marc Vidal    35      26-40                      3   
2           3    Ana López    29      26-40                      2   

                                  segmento  factor_codigo  \
0    Cesta abandonada hace más de 24 horas           0.90   
1  Cesta abandonada hace menos de 24 horas           0.75   
2           Llega al checkout pero no paga           0.50   

                                             mensaje  factor_edad  \
0  Te echamos de menos. Recupera tu cesta ahora y...          1.2   
1  ¡No olvides tu cesta! Completa tu compra antes...          1.0   
2  Finaliza tu pedido ahora y disfruta de una exp...          1.0   

   riesgo_final  
0          1.08  
1          0.75  
2          0.50  


In [34]:
# creamos la función principal que aglutine y orqueste el resto de funciones

def main_marketing(lista_clientes):

    clientes_procesados = procesar_clientes(lista_clientes)

    df_clientes_procesados = convertir_dataframe(clientes_procesados)

    return df_clientes_procesados

In [35]:
# guardamos la información prcesada en un DataFrame

df_clientes = main_marketing(clientes_marketing)

In [36]:
# para responder a las preguntas de negocio ordenaremos el dataframe por 'riesgo_final' de mayor a menor

df_clientes.sort_values('riesgo_final', ascending=False)

,id_cliente,nombre,edad,grupo_edad,codigo_comportamiento,segmento,factor_codigo,mensaje,factor_edad,riesgo_final
0,1,Laura Serra,23,18-25,4,Cesta abandonada hace más de 24 horas,0.90,Te echamos de menos. Recupera tu cesta ahora y...,1.20,1.0800
9,10,Núria Costa,33,26-40,4,Cesta abandonada hace más de 24 horas,0.90,Te echamos de menos. Recupera tu cesta ahora y...,1.00,0.9000
4,5,Mónica Pérez,47,41-60,4,Cesta abandonada hace más de 24 horas,0.90,Te echamos de menos. Recupera tu cesta ahora y...,0.85,0.7650
1,2,Marc Vidal,35,26-40,3,Cesta abandonada hace menos de 24 horas,0.75,¡No olvides tu cesta! Completa tu compra antes...,1.00,0.7500
3,4,Joan Riera,19,18-25,2,Llega al checkout pero no paga,0.50,Finaliza tu pedido ahora y disfruta de una exp...,1.20,0.6000
6,7,Pilar Sánchez,61,60+,3,Cesta abandonada hace menos de 24 horas,0.75,¡No olvides tu cesta! Completa tu compra antes...,0.70,0.5250
2,3,Ana López,29,26-40,2,Llega al checkout pero no paga,0.50,Finaliza tu pedido ahora y disfruta de una exp...,1.00,0.5000
8,9,Diego Romero,24,18-25,1,Añade productos pero no llega al checkout,0.25,Tus productos siguen esperándote. Completa tu ...,1.20,0.3000
5,6,Luis García,52,41-60,1,Añade productos pero no llega al checkout,0.25,Tus productos siguen esperándote. Completa tu ...,0.85,0.2125
7,8,Eva Martín,38,26-40,0,Navega pero no añade productos,0.10,Descubre nuestros productos más populares y en...,1.00,0.1000


In [37]:
# función para poder sacar la respuesta a las preguntas de Marketing

def mostrar_respuestas(df_clientes):

    # ordenamos el DataFrame por riesgo_final de manera descendente y nos quedamos con los 3 primeros
    print("1. Clientes a contactar primero:")
    print(
        df_clientes.sort_values(
            by="riesgo_final",
            ascending=False
        )[["nombre", "riesgo_final"]].head(3)
    )

    # ordenamos el DataFrame por codigo_comportamiento y factor_edad para poder compararlos mejor entre ellos
    print("\n2. Comparación por comportamiento y edad:")
    print(
        df_clientes.sort_values(
            by=["codigo_comportamiento", "factor_edad"],
            ascending=[True, False]
        )[[
            "nombre",
            "codigo_comportamiento",
            "grupo_edad",
            "factor_edad",
            "riesgo_final"
        ]]
    )

    # ordenamos el DataFrame por riesgo_final de manera ascendente y nos quedamos con los 3 primeros
    print("\n3. Clientes con menor interés:")
    print(
        df_clientes.sort_values(
            by="riesgo_final"
        )[["nombre", "segmento", "riesgo_final"]].head(3)
    )

    # ordenamos el DataFrame por riesgo_final de manera descendente y nos quedamos con el primer cliente
    print("\n4. Cliente más rentable de recuperar:")
    print(
        df_clientes.sort_values(
            by="riesgo_final",
            ascending=False
        )[["nombre", "segmento", "riesgo_final"]].head(1)
    )

In [38]:
# llamamos a la función para que nos muestre las respuestas respecto a nuestro DataFrame

mostrar_respuestas(df_clientes)

1. Clientes a contactar primero:
         nombre  riesgo_final
0   Laura Serra         1.080
9   Núria Costa         0.900
4  Mónica Pérez         0.765

2. Comparación por comportamiento y edad:
          nombre  codigo_comportamiento grupo_edad  factor_edad  riesgo_final
7     Eva Martín                      0      26-40         1.00        0.1000
8   Diego Romero                      1      18-25         1.20        0.3000
5    Luis García                      1      41-60         0.85        0.2125
3     Joan Riera                      2      18-25         1.20        0.6000
2      Ana López                      2      26-40         1.00        0.5000
1     Marc Vidal                      3      26-40         1.00        0.7500
6  Pilar Sánchez                      3        60+         0.70        0.5250
0    Laura Serra                      4      18-25         1.20        1.0800
9    Núria Costa                      4      26-40         1.00        0.9000
4   Mónica Pérez        

1. Qué clientes se han de contactar primero en una campaña de retargeting?

Aquellos clientes con mayor puntuación en riesgo_final, en nuestro caso, Laura Serra, Núria Costa, y Mónica Pérez.

2. Que diferencias hay entre clientes con el mismo comportamiento pero edades diferentes?

Si comparamos clientes que tienen el mismo comportamiento, pero que difieren en la edad,
podemos ver que los clientes más jóvenen tendrán más riesgo_final que aquellos más mayores.

3. Qué clientes presentan tan poco interés que no merece la pena invertir presupuesto?

Eva Martín, Luís García, y Diego Romero, ya que son los que menor riesgo_final presentan.

En esencia, aquellos clientes que menos interactúen con la página (menor código de comportamiento), y a la vez sean más mayores.

4. Qué cliente sería el más rentable de recuperar?

Laura Serra, ya que tiene el mayor 'riesgo_final', producto de haber llenado una cesta hace más de 24h, y ser parte de un grupo de edad joven y teóricamente más impulsivo/reactivo.

################################################################

# Caso 3: Gestión de riesgo y priorización de pacientes

Trabajas como analista de datos en un centro médico especializado. Varios pacientes han enviado sus síntomas a través de un formulario online, pero los datos llegan en un formato desordenado y poco estructurado, y el equipo médico necesita que los organices para poder priorizar el orden de atención según la gravedad.

A continuación tienes una lista con información real procedente del sistema:

pacientes = [
    "Maria|45|tos, Fiebre Alta, dolor_pecho",
    "Luis|33|dolor_cabeza,faTiga",
    "Sara|67|fiebre alta, dificultad RESPIRAR, tos",
    "Jordi|52|fatiga , tos",
    "Anna|29|fiebre alta,dolor_cabeza"
]

El departamento médico también te ha proporcionado el siguiente diccionario de niveles de riesgo por síntoma:

niveles_riesgo = {
    "fiebre_alta": 3,
    "dificultad_respirar": 5,
    "dolor_pecho": 4,
    "tos": 1,
    "fatiga": 1,
    "dolor_cabeza": 1
}

El objetivo es transformar estos datos en información útil para los médicos. A partir de los registros en bruto, debes:

- Convertir cada cadena de texto en una estructura limpia (diccionario o similar).
- Separar y normalizar la lista de síntomas:
  - minúsculas
  - eliminar espacios
  - dividir correctamente por comas
- Asignar un nivel de riesgo a cada síntoma utilizando el diccionario nivells_risc.
- Calcular el riesgo total de cada paciente sumando los valores.
- Ordenar los pacientes de mayor a menor riesgo para determinar la prioridad de atención.



## Para resolver el problema:

1. Analizar el formato de entrada para identificar cómo están separados los datos de cada paciente
2. Convertir cada registro de texto en una estructura organizada (diccionario) con nombre, edad y síntomas
3. Separar la cadena de síntomas en una lista individual de síntomas
4. Normalizar los síntomas convirtiéndolos a minúsculas, eliminando espacios innecesarios y sustituyendo espacios internos por guiones bajos para unificar el formato
5. Asignar a cada síntoma su nivel de riesgo utilizando el diccionario proporcionado por el departamento médico
6. Calcular el riesgo total de cada paciente sumando los niveles de riesgo de todos sus síntomas
7. Añadir el riesgo total a la información de cada paciente
8. Ordenar los pacientes de mayor a menor riesgo para establecer la prioridad de atención
9. Convertir la información procesada en un DataFrame para facilitar la visualización y el análisis
10. Mostrar el listado final de pacientes ordenado según su prioridad médica

In [39]:
# la lista de pacientes tiene un patrón de {nombre} | {edad} | {síntomas}
# podemos separar la información con un .split("|"), así aislaremos los síntomas
# para limpiar los síntomas tendremos haremos un .split por ","
# después eliminaremos espacios en los extremos con .strip(), pasamos todo a minúsculas con .lower(),
# y finalmente reemplazamos los espacios por "_" con replace() para que queden normalizados y estandarizados

pacientes = [
    "Maria|45|tos, Fiebre Alta, dolor_pecho",
    "Luis|33|dolor_cabeza,faTiga",
    "Sara|67|fiebre alta, dificultad RESPIRAR, tos",
    "Jordi|52|fatiga , tos",
    "Anna|29|fiebre alta,dolor_cabeza"
]

In [40]:
# diccionario de niveles de riesgo

niveles_riesgo = {
    "fiebre_alta": 3,
    "dificultad_respirar": 5,
    "dolor_pecho": 4,
    "tos": 1,
    "fatiga": 1,
    "dolor_cabeza": 1
}

In [41]:
# función para normalizar los datos, separarlos por "|" y crear la nueva lista con diccionario de pacientes

def normalizar_datos(lista_pacientes):

    pacientes = []

    for paciente in lista_pacientes:

        paciente = paciente.split("|")

        pacientes.append({
            "nombre": paciente[0],
            "edad": paciente[1],
            "sintomas": paciente[2]
        })

    return pacientes

In [42]:
# creamos una lista para probar las funciones copiando pacientes

pacientes_prueba = pacientes

# aplicamos la función a la lista de prueba pacientes_prueba

pacientes_prueba = normalizar_datos(pacientes_prueba)

# vemos cómo queda el primer paciente de la lista de prueba

print(pacientes_prueba[0])

{'nombre': 'Maria', 'edad': '45', 'sintomas': 'tos, Fiebre Alta, dolor_pecho'}


In [43]:
# función para normalizar los síntomas separándolos por coma para tenerlos separados

def normalizar_sintomas(lista_pacientes):

    for paciente in lista_pacientes:

        paciente['sintomas'] = paciente['sintomas'].split(",")

    return lista_pacientes

In [44]:
# aplicamos la función a la lista de prueba pacientes_prueba

pacientes_prueba = normalizar_sintomas(pacientes_prueba)

# vemos cómo queda el primer paciente de la lista de prueba

print(pacientes_prueba[0])

{'nombre': 'Maria', 'edad': '45', 'sintomas': ['tos', ' Fiebre Alta', ' dolor_pecho']}


In [45]:
# con esta función, una vez separados los síntomas, eliminamos los espacios en los extremos,
# pasamos todo a minúsculas y reemplazamos los espacios entre palabras por "_" para tenerlos estandarizados

def procesar_sintomas(lista_pacientes):

    for paciente in lista_pacientes:

        sintomas_limpios = []

        for sintoma in paciente['sintomas']:

            sintoma = sintoma.strip().lower().replace(" ", "_")

            sintomas_limpios.append(sintoma)

        paciente['sintomas'] = sintomas_limpios

    return lista_pacientes

In [46]:
# aplicamos la función a la lista de prueba pacientes_prueba

pacientes_prueba = procesar_sintomas(pacientes_prueba)

# vemos cómo queda el primer paciente de la lista de prueba

print(pacientes_prueba[0])

{'nombre': 'Maria', 'edad': '45', 'sintomas': ['tos', 'fiebre_alta', 'dolor_pecho']}


In [47]:
# función para calcular el riesgo según la fórmula que nos facilitan y añadir la llave riesgo al diccionario de cada paciente

def calcular_riesgo(lista_pacientes):

    for paciente in lista_pacientes:

        paciente['riesgo'] = 0

        for sintoma in paciente['sintomas']:

            paciente['riesgo'] += niveles_riesgo[sintoma]

    return lista_pacientes

In [48]:
# aplicamos la función a la lista de prueba pacientes_prueba

pacientes_prueba = calcular_riesgo(pacientes_prueba)

# vemos cómo queda el primer paciente de la lista de prueba

print(pacientes_prueba[0])

{'nombre': 'Maria', 'edad': '45', 'sintomas': ['tos', 'fiebre_alta', 'dolor_pecho'], 'riesgo': 8}


In [49]:
# función para convertir la información a un DataFrame

def convertir_dataframe(lista_pacientes):

    df_pacientes = pd.DataFrame(lista_pacientes)

    return df_pacientes

In [50]:
# aplicamos la función a la lista de prueba pacientes_prueba

df_pacientes_prueba = convertir_dataframe(pacientes_prueba)

# vemos cómo quedan los primeros 3 pacientes del DataFrame de prueba

print(df_pacientes_prueba.head(3))

  nombre edad                                 sintomas  riesgo
0  Maria   45          [tos, fiebre_alta, dolor_pecho]       8
1   Luis   33                   [dolor_cabeza, fatiga]       2
2   Sara   67  [fiebre_alta, dificultad_respirar, tos]       9


In [51]:
# función para ordenar el DataFrame en base al riesgo de los pacientes, de mayor a menor

def ordenar_pacientes(df_pacientes):

    pacientes_ordenados = df_pacientes.sort_values(
        by="riesgo",
        ascending=False
    )

    return pacientes_ordenados

In [52]:
# aplicamos la función al DataFrame de prueba df_pacientes_prueba

df_pacientes_prueba = ordenar_pacientes(df_pacientes_prueba)

# vemos cómo quedan los primeros 3 pacientes del DataFrame de prueba ordenados

print(df_pacientes_prueba.head(3))

  nombre edad                                 sintomas  riesgo
2   Sara   67  [fiebre_alta, dificultad_respirar, tos]       9
0  Maria   45          [tos, fiebre_alta, dolor_pecho]       8
4   Anna   29              [fiebre_alta, dolor_cabeza]       4


In [53]:
# aglutinamos todas demás funciones en la función principal para orquestarlas, devolviendo al final un df ordenado por riesgo de mayor a menor

def main_ambulatorio(lista_pacientes):

    pacientes = normalizar_datos(lista_pacientes)

    pacientes = normalizar_sintomas(pacientes)

    pacientes = procesar_sintomas(pacientes)

    pacientes = calcular_riesgo(pacientes)

    df_pacientes = convertir_dataframe(pacientes)

    pacientes_ordenados = ordenar_pacientes(df_pacientes)

    return pacientes_ordenados

In [54]:
df_pacientes = main_ambulatorio(pacientes)

df_pacientes

,nombre,edad,sintomas,riesgo
2,Sara,67,"[fiebre_alta, dificultad_respirar, tos]",9
0,Maria,45,"[tos, fiebre_alta, dolor_pecho]",8
4,Anna,29,"[fiebre_alta, dolor_cabeza]",4
1,Luis,33,"[dolor_cabeza, fatiga]",2
3,Jordi,52,"[fatiga, tos]",2


################################################################

# Caso 4: Clasificación de clientes de seguros

Trabajas como analista de datos en una compañía aseguradora. La empresa quiere calcular el riesgo de los clientes, el precio estimado de la póliza y la probabilidad de fraude, utilizando un código de comportamiento que ha sido generado por un modelo interno.

Tenemos la siguiente lista de clientes:

clientes_seguros = [
    {'nombre': 'Ana', 'edad': 42, 'codigo': 0},
    {'nombre': 'Carlos', 'edad': 30, 'codigo': 2},
    {'nombre': 'Isabel', 'edad': 55, 'codigo': 3},
    {'nombre': 'Jorge', 'edad': 40, 'codigo': 1},
    {'nombre': 'Marta', 'edad': 28, 'codigo': 0}
]

Y este diccionario con las categorías del sistema:

niveles_seguro = {
    0: {"categoria": "Bajo riesgo", "precio_base": 120, "fraude": 0.01},
    1: {"categoria": "Riesgo medio", "precio_base": 200, "fraude": 0.05},
    2: {"categoria": "Riesgo alto", "precio_base": 350, "fraude": 0.15},
    3: {"categoria": "Riesgo crítico", "precio_base": 500, "fraude": 0.30}
}

El precio final se debe calcular con la fórmula:

precio_final = precio_base * (1 + edad / 100)

Clasifica cada cliente utilizando niveles_seguro.

Enriquece con la categoría, precio base, y probabilidad de fraude.

Calcula el precio final de la póliza con la fórmula facilitada.

Crea un DataFrame con la información final.

Ordena los clientes por probabilidad de fraude, y también por precio final.

## Para resolver el problema:

1. Obtener la información adicional de cada cliente a partir del código de comportamiento
2. Añadir categoría, precio base y probabilidad de fraude a cada cliente
3. Calcular el precio final de la póliza
4. Procesar todos los clientes mediante una función principal
5. Convertir los resultados a un DataFrame
6. Analizar los clientes según fraude y precio final
7. Mostrar el informe final y las conclusiones

In [55]:
# información de clientes y niveles de seguro

clientes_seguros = [
    {'nombre': 'Ana', 'edad': 42, 'codigo': 0},
    {'nombre': 'Carlos', 'edad': 30, 'codigo': 2},
    {'nombre': 'Isabel', 'edad': 55, 'codigo': 3},
    {'nombre': 'Jorge', 'edad': 40, 'codigo': 1},
    {'nombre': 'Marta', 'edad': 28, 'codigo': 0}
]

niveles_seguro = {
    0: {"categoria": "Bajo riesgo", "precio_base": 120, "fraude": 0.01},
    1: {"categoria": "Riesgo medio", "precio_base": 200, "fraude": 0.05},
    2: {"categoria": "Riesgo alto", "precio_base": 350, "fraude": 0.15},
    3: {"categoria": "Riesgo crítico", "precio_base": 500, "fraude": 0.30}
}

In [56]:
# función para enriquecer la lista de clientes con categoría, precio_base y fraude en base a la información en niveles_seguro

def enriquecer_clientes(lista_clientes):

    for cliente in lista_clientes:
        cliente['categoria'] = niveles_seguro[cliente['codigo']]['categoria']
        cliente['precio_base'] = niveles_seguro[cliente['codigo']]['precio_base']
        cliente['fraude'] = niveles_seguro[cliente['codigo']]['fraude']

    return lista_clientes

In [57]:
# creamos una lista para probar las funciones copiando clientes_seguros

clientes_seguros_prueba = clientes_seguros

# aplicamos la función a la lista de prueba clientes_seguros

clientes_seguros_prueba = enriquecer_clientes(clientes_seguros_prueba)

# vemos cómo queda el primer cliente de la lista de prueba

print(clientes_seguros_prueba[0])

{'nombre': 'Ana', 'edad': 42, 'codigo': 0, 'categoria': 'Bajo riesgo', 'precio_base': 120, 'fraude': 0.01}


In [58]:
# función para calcular el precio final con la fórmula proporcionada

def calcular_precio_final(lista_clientes):

    for cliente in lista_clientes:

        cliente['precio_final'] = round((cliente['precio_base'] * (1 + cliente['edad'] / 100)), 2)

    return lista_clientes

In [59]:
# aplicamos la función a la lista de prueba clientes_seguros

clientes_seguros_prueba = calcular_precio_final(clientes_seguros_prueba)

# vemos cómo queda el primer cliente de la lista de prueba

print(clientes_seguros_prueba[0])

{'nombre': 'Ana', 'edad': 42, 'codigo': 0, 'categoria': 'Bajo riesgo', 'precio_base': 120, 'fraude': 0.01, 'precio_final': 170.4}


In [60]:
# función para convertir la información a un DataFrame

def convertir_dataframe(lista_clientes):

    df_clientes = pd.DataFrame(lista_clientes)

    return df_clientes

In [61]:
# aplicamos la función a la lista de prueba clientes_seguros

df_clientes_seguros_prueba = convertir_dataframe(clientes_seguros_prueba)

# vemos cómo quedan los primeros 3 clientes del DataFrame de prueba

print(df_clientes_seguros_prueba.head(3))

   nombre  edad  codigo       categoria  precio_base  fraude  precio_final
0     Ana    42       0     Bajo riesgo          120    0.01         170.4
1  Carlos    30       2     Riesgo alto          350    0.15         455.0
2  Isabel    55       3  Riesgo crítico          500    0.30         775.0


In [62]:
# juntamos las funciones en una función principal para orquestarlas

def main_seguros(lista_clientes):

    clientes = enriquecer_clientes(lista_clientes)

    clientes = calcular_precio_final(clientes)

    df_clientes = convertir_dataframe(clientes)

    return df_clientes

In [63]:
# guardamos la información en un DataFrame

df_clientes = main_seguros(clientes_seguros)

1. Qué cliente tiene más probabilidades de cometer fraude?

In [64]:
# ordenamos el df en función del fraude y nos quedamos con el primer resultado

df_clientes.sort_values('fraude', ascending=False).head(1)

,nombre,edad,codigo,categoria,precio_base,fraude,precio_final
2,Isabel,55,3,Riesgo crítico,500,0.3,775.0


2. Quién tiene el precio final más alto?

In [65]:
# ordenamos el df en función del precio final y nos quedamos con el primer resultado

df_clientes.sort_values('precio_final', ascending=False).head(1)

,nombre,edad,codigo,categoria,precio_base,fraude,precio_final
2,Isabel,55,3,Riesgo crítico,500,0.3,775.0


3. Quién sería el prioritario a revisar manualmente?

In [66]:
# creamos una nueva columna 'indice_revision' en base al producto de fraude y precio final, y esto nos dará un índice de priorización a revisar

df_clientes['indice_revision'] = (
    df_clientes['fraude'] * df_clientes['precio_final']
)

df_clientes.sort_values(
    "indice_revision",
    ascending=False
).head(3)

,nombre,edad,codigo,categoria,precio_base,fraude,precio_final,indice_revision
2,Isabel,55,3,Riesgo crítico,500,0.30,775.0,232.50
1,Carlos,30,2,Riesgo alto,350,0.15,455.0,68.25
3,Jorge,40,1,Riesgo medio,200,0.05,280.0,14.00


Cliente prioritario para revisión manual:

Isabel, ya que combina la mayor probabilidad de fraude con el mayor importe económico, representando el mayor riesgo potencial para la compañía

################################################################

# Caso 5: Generador de contraseñas seguras

Trabajas como analista de datos dentro del departamento de Ciberseguridad de una empresa tecnológica. Se ha detectado que la mayoría de los trabajadores siguen utilizando contraseñas muy débiles como “Asdf1234”, fechas de cumpleaños o combinaciones fáciles de adivinar. Por este motivo, el responsable del departamento te ha pedido desarrollar un generador de contraseñas seguras, totalmente parametrizable y modular.

Requisitos del generador:

Debes crear una función principal y todas las subfunciones necesarias para:

- Generar contraseñas de longitud variable.
- Decidir si deben incluir:
  - Mayúsculas
  - Minúsculas
  - Números
  - Signos especiales
- Asegurarte de que aparezca al menos un carácter de cada tipo seleccionado.
- Aleatorizar completamente la contraseña.
- Devolver la contraseña generada.

## Para resolver el problema:

1. Crear un diccionario con los caracteres disponibles para cada categoría
2. Validar los parámetros de entrada
3. Comprobar que la longitud sea un número entero válido
4. Comprobar que al menos una categoría de caracteres esté activada
5. Identificar qué categorías ha seleccionado el usuario
6. Crear una lista con los grupos de caracteres permitidos
7. Añadir un carácter aleatorio de cada grupo seleccionado para garantizar que aparezca al menos una vez en la contraseña
8. Completar la contraseña con caracteres aleatorios elegidos entre los grupos disponibles hasta alcanzar la longitud solicitada
9. Mezclar aleatoriamente todos los caracteres generados para evitar patrones previsibles
10. Unir los caracteres en una única cadena de texto
11. Devolver la contraseña generada

In [67]:
# creamos un diccionario de caracteres a utilizar con la librería string

diccionario_caracteres = {
    "mayusculas": string.ascii_uppercase,
    "minusculas": string.ascii_lowercase,
    "numeros": string.digits,
    "simbolos": string.punctuation
}

In [68]:
# función para validar que num sea un entero

def validar_longitud(num):
    try:
        return int(num)

    except ValueError:
        raise ValueError("Debes introducir un número entero")

In [69]:
# aplicamos la función para validar diferentes posibles entradas

print(validar_longitud(12))
print(validar_longitud("12"))

# si ejecutamos la siguiente línea de código nos avisará de un error al no poder convertir el string a int
# print(validar_longitud("prueba"))

12
12


In [70]:
# función para definir los grupos de caracteres disponibles para crear la contraseña

def definir_grupos_disponibles(mayusculas, minusculas, numeros, simbolos):

    grupos_disponibles = []

    if mayusculas == True:
        grupos_disponibles.append(diccionario_caracteres["mayusculas"])

    if minusculas == True:
        grupos_disponibles.append(diccionario_caracteres["minusculas"])

    if numeros == True:
        grupos_disponibles.append(diccionario_caracteres["numeros"])

    if simbolos == True:
        grupos_disponibles.append(diccionario_caracteres["simbolos"])

    return grupos_disponibles

In [71]:
# llamamos a la función y guardamos el resultado en grupos_disponibles_prueba

grupos_disponibles_prueba = definir_grupos_disponibles(mayusculas=True, minusculas=True, numeros=True, simbolos=True)

# mostramos los caracteres disponibles de grupos_disponibles_prueba

print(grupos_disponibles_prueba)

['ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz', '0123456789', '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~']


In [72]:
# función para añadir al menos un caracter de los caracteres disponibles a la contraseña
# ésto nos ahorra tener que hacer una validación final de caracteres mínimos

def caracteres_minimos(grupos_disponibles):

    contrasena = []

    for grupo in grupos_disponibles:

        contrasena.append(random.choice(list(grupo)))

    return contrasena

In [73]:
# llamamos a la función y guardamos un caracter de cada grupo disponible en grupos_disponibles_prueba

contrasena_prueba = caracteres_minimos(grupos_disponibles_prueba)

# mostramos los caracteres de contrasena_prueba

print(contrasena_prueba)

[np.str_('I'), np.str_('z'), np.str_('1'), np.str_('`')]


In [74]:
# función para rellenar la contraseña con caracteres aleatorios de los disponibles hasta la longitud determinada

def resto_caracteres(contrasena, num, grupos_disponibles):

    while len(contrasena) < num:

        caracteres = random.choice(grupos_disponibles)

        caracter = random.choice(list(caracteres))

        contrasena.append(caracter)

    return contrasena

In [75]:
# llamamos a la función que rellena contrasena_prueba con caracteres aleatorios hasta llegar al número definido en num

contrasena_prueba = resto_caracteres(contrasena_prueba, 12, grupos_disponibles_prueba)

# mostramos los caracteres de contrasena_prueba

print(contrasena_prueba)

[np.str_('I'), np.str_('z'), np.str_('1'), np.str_('`'), np.str_('{'), np.str_('4'), np.str_('B'), np.str_('^'), np.str_('['), np.str_('j'), np.str_('I'), np.str_('L')]


In [76]:
# función para aleatorizar el orden de los caracteres para evitar patrones recurrentes

def aleatorizar_contrasena(contrasena):

    random.shuffle(contrasena)

    return contrasena

In [77]:
# aleatorizamos el orden de caracteres en contrasena_prueba

contrasena_prueba = aleatorizar_contrasena(contrasena_prueba)

# mostramos los caracteres de contrasena_prueba después de la aleatorización

print(contrasena_prueba)

[np.str_('B'), np.str_('z'), np.str_('['), np.str_('^'), np.str_('1'), np.str_('I'), np.str_('L'), np.str_('4'), np.str_('{'), np.str_('`'), np.str_('j'), np.str_('I')]


In [78]:
# función para unir los caracteres en una cadena de texto

def convertir_string(contrasena):

    contrasena = "".join(contrasena)

    return contrasena

In [79]:
# convertimos la lista contrasena_prueba a un string

contrasena_prueba = convertir_string(contrasena_prueba)

# mostramos la contrasena_prueba final

print(contrasena_prueba)

Bz[^1IL4{`jI


In [80]:
# funcion principal anidando todas las funciones y añadiendo validaciones para orquestarlo todo

def main_generador_contrasena(num, mayusculas=True, minusculas=True, numeros=True, simbolos=False):

    # validamos que num sea entero y sea mayor o igual que 8

    num = validar_longitud(num)

    # añadimos una validación para que la contraseña sea de por lo menos 8 caracteres
    # esto nos permitirá no tener que validar para números negativos ni para cantidad de grupos mínimos
    if num < 8:
        raise ValueError(f"La longitud mínima debe ser 8 caracteres")

    # validamos que al menos un tipo de caracteres sea True

    if not any([mayusculas, minusculas, numeros, simbolos]):
        raise ValueError("Debe seleccionarse al menos un tipo de carácter")

    grupos_disponibles = definir_grupos_disponibles(mayusculas, minusculas, numeros, simbolos)

    contrasena = caracteres_minimos(grupos_disponibles)

    contrasena = resto_caracteres(contrasena, num, grupos_disponibles)

    contrasena = aleatorizar_contrasena(contrasena)

    contrasena = convertir_string(contrasena)

    return contrasena

In [81]:
palabra_secreta = main_generador_contrasena(12, simbolos=True)

print(palabra_secreta)

1'91,F>C2c.)


################################################################

# Caso 6: Procesamiento automático de datos deportivos

Ahora formas parte de un equipo de analítica dentro de una organización deportiva que gestiona datos de la liga catalana. Tu compañera de trabajo quiere automatizar la lectura de su archivo histórico de resultados y extraer las estadísticas más importantes. Te ha pedido crear un pequeño sistema en Python que procese el archivo "historic partits.txt".

Mediante un código modular, claro y documentado, tu programa debe calcular:

1. Qué equipo es el más goleador.

2. Qué equipo es el más goleado.

3. El número total de goles que ha marcado cada equipo.

4. La clasificación final ordenada por puntos, donde:

- Victoria = 3 puntos
- Empate = 1 punto
- Derrota = 0 puntos

En este nivel no se proporciona ninguna función. Eres tú quien debe diseñar un flujo completo. El código debe reflejar una buena organización interna.

El resultado que espera el equipo debe ser un DataFrame o diccionario final similar a una tabla de clasificación.

Además del texto:

Equipo más goleador: _______

Equipo más goleado: ________

Clasificación final:
1. _______
2. _______
...

## Para resolver el problema:

1. Leer el archivo .txt y almacenar cada partido en una lista
2. Separar la información de cada partido en sus componentes:
- Equipo local
- Resultado
- Equipo visitante
3. Separar los goles del resultado y convertirlos a valores numéricos
4. Identificar todos los equipos participantes
5. Crear una estructura de datos para almacenar las estadísticas de cada equipo:
- Goles a favor
- Goles en contra
- Partidos ganados
- Partidos empatados
- Partidos perdidos
- Puntos
6. Recorrer todos los partidos y actualizar las estadísticas correspondientes de cada equipo
7. Calcular la clasificación final a partir de los puntos obtenidos
8. Convertir la información a un DataFrame para facilitar el análisis
9. Identificar:
- El equipo más goleador
- El equipo más goleado
- La clasificación final ordenada por puntos
10. Mostrar el resumen final y la tabla de clasificación

In [82]:
# guardamos la información del documento en lista_partidos

with open("/content/drive/MyDrive/historic partits.txt", encoding="utf-8") as f:
    lista_partidos = [line.strip() for line in f]

In [83]:
# función para procesar cada línea de la lista y procesarla, extrayendo:
# [equipo_local, resultado, equipo_visitante]

def procesar_partidos(lista_partidos):

    partidos_procesados = []

    for partido in lista_partidos:

        elemento = partido.split("\t")

        partidos_procesados.append(elemento)

    return partidos_procesados

In [84]:
# creamos una lista de prueba copiando lista_partidos

lista_partidos_prueba = lista_partidos

# llamamos a la función y guardamos el resultado en lista_partidos_prueba_procesados

lista_partidos_prueba_procesados = procesar_partidos(lista_partidos_prueba)

# mostramos el primer índice de lista_partidos_prueba_procesados

print(lista_partidos_prueba_procesados[0])

['Manlleu', '0-1', 'Granollers']


In [85]:
# función para convertir los marcadores "X-Y" en una lista de enteros [X,Y] y poder así realizar cálculos

def normalizar_resultados(partidos_procesados):

    for partido in partidos_procesados:

        partido[1] = partido[1].split("-")
        partido[1][0] = int(partido[1][0])
        partido[1][1] = int(partido[1][1])

    return partidos_procesados

In [86]:
# llamamos a la función y guardamos el resultado

lista_partidos_prueba_procesados = normalizar_resultados(lista_partidos_prueba_procesados)

# mostramos el primer índice de lista_partidos_prueba_procesados

print(lista_partidos_prueba_procesados[0])

['Manlleu', [0, 1], 'Granollers']


In [87]:
# función para crear un diccionario vacío por cada equipo

def equipos_participantes(partidos_procesados):

    estadisticas_equipos = {}

    for partido in partidos_procesados:

        estadisticas_equipos[partido[0]] = {}

        estadisticas_equipos[partido[2]] = {}

    return estadisticas_equipos

In [88]:
# creamos un diccionario de estadísticas con una entrada para cada equipo

estadisticas_equipos_prueba = equipos_participantes(lista_partidos_prueba_procesados)

# mostramos el diccionario con los equipos registrados

print(estadisticas_equipos_prueba)

{'Manlleu': {}, 'Granollers': {}, 'Vilafranca': {}, 'Terrassa': {}, 'Olot': {}, 'Cornellà': {}, 'Prat': {}, 'Cerdanyola': {}, 'Sant Andreu': {}, 'FC Barcelona': {}, 'Badalona': {}, 'Sabadell': {}, 'Europa': {}, 'Llagostera': {}, 'Figueres': {}, 'RCD Espanyol': {}, 'Reus Deportiu': {}, 'Nàstic de Tarragona': {}, 'Lleida Esportiu': {}, 'Girona FC': {}}


In [89]:
# función para introducir las llaves con los valores a 0 que queremos contabilizar

def inicializar_metricas(estadisticas_equipos):

    for equipo in estadisticas_equipos:
        estadisticas_equipos[equipo]['goles_favor'] = 0
        estadisticas_equipos[equipo]['goles_contra'] = 0
        estadisticas_equipos[equipo]['ganados'] = 0
        estadisticas_equipos[equipo]['empatados'] = 0
        estadisticas_equipos[equipo]['perdidos'] = 0
        estadisticas_equipos[equipo]['puntos'] = 0

    return estadisticas_equipos

In [90]:
# introducimos en cada equipo una llave para cada métrica que vamos a contabilizar

estadisticas_equipos_prueba = inicializar_metricas(estadisticas_equipos_prueba)

# mostramos el diccionario creado para el equipo 'Manlleu'

print(estadisticas_equipos_prueba['Manlleu'])

{'goles_favor': 0, 'goles_contra': 0, 'ganados': 0, 'empatados': 0, 'perdidos': 0, 'puntos': 0}


In [91]:
# función en la que introducimos la lógica para calcular resultados de partidos (goles, partidos y puntos)

def contabilizar_resultados(estadisticas_equipos, partidos_procesados):

    for partido in partidos_procesados:

        estadisticas_equipos[partido[0]]['goles_favor'] += partido[1][0]
        estadisticas_equipos[partido[0]]['goles_contra'] += partido[1][1]
        estadisticas_equipos[partido[2]]['goles_favor'] += partido[1][1]
        estadisticas_equipos[partido[2]]['goles_contra'] += partido[1][0]
        if partido[1][0] > partido[1][1]:
            estadisticas_equipos[partido[0]]['ganados'] += 1
            estadisticas_equipos[partido[0]]['puntos'] += 3
            estadisticas_equipos[partido[2]]['perdidos'] += 1
        elif partido[1][0] < partido[1][1]:
            estadisticas_equipos[partido[0]]['perdidos'] += 1
            estadisticas_equipos[partido[2]]['ganados'] += 1
            estadisticas_equipos[partido[2]]['puntos'] += 3
        else:
            estadisticas_equipos[partido[0]]['empatados'] += 1
            estadisticas_equipos[partido[2]]['empatados'] += 1
            estadisticas_equipos[partido[0]]['puntos'] += 1
            estadisticas_equipos[partido[2]]['puntos'] += 1

    return estadisticas_equipos

In [92]:
# aplicamos la función para contabilizar los resultados de la lista lista_partidos_prueba_procesados
# y así actualizar los valores de las métricas a contabilizar

estadisticas_equipos_prueba = contabilizar_resultados(estadisticas_equipos_prueba, lista_partidos_prueba_procesados)

# mostramos el diccionario actualizado del equipo 'Manlleu'

print(estadisticas_equipos_prueba['Manlleu'])

{'goles_favor': 106, 'goles_contra': 118, 'ganados': 16, 'empatados': 9, 'perdidos': 18, 'puntos': 57}


In [93]:
# función para convertir a DataFrame y aplicar .T para transposicionar

def convertir_dataframe(estadisticas_equipos):

    df_partidos = pd.DataFrame(estadisticas_equipos)

    df_partidos = df_partidos.T

    return df_partidos

In [94]:
# llamamos a la función para convertir el diccionario a DataFrame y transposicionarlo

df_equipos_prueba = convertir_dataframe(estadisticas_equipos_prueba)

# mostramos los primeros 3 resultados del DataFrame sin orden específico

print(df_equipos_prueba.head(3))

            goles_favor  goles_contra  ganados  empatados  perdidos  puntos
Manlleu             106           118       16          9        18      57
Granollers          122           117       21          4        24      67
Vilafranca          157           172       20         12        25      72


In [95]:
# función para mostrar el resumen de la temporada

def mostrar_resumen(df_partidos):

    equipo_mas_goleador = df_partidos["goles_favor"].idxmax()

    equipo_mas_goleado = df_partidos["goles_contra"].idxmax()

    clasificacion = df_partidos.sort_values(
        by="puntos",
        ascending=False
    )

    print(f"Equipo más goleador: {equipo_mas_goleador}")
    print(f"Equipo más goleado: {equipo_mas_goleado}")

    print("\nClasificación final:")

    for posicion, equipo in enumerate(clasificacion.index, start=1):

        print(f"{posicion}. {equipo}")

In [96]:
# llamamos a la función para que nos enseñe por pantalla el equipo más goleador, el más goleado, y la clasificación final

mostrar_resumen(df_equipos_prueba)

Equipo más goleador: Figueres
Equipo más goleado: Vilafranca

Clasificación final:
1. Girona FC
2. Llagostera
3. Sabadell
4. Cornellà
5. RCD Espanyol
6. Figueres
7. Lleida Esportiu
8. Terrassa
9. FC Barcelona
10. Vilafranca
11. Badalona
12. Nàstic de Tarragona
13. Reus Deportiu
14. Granollers
15. Olot
16. Sant Andreu
17. Manlleu
18. Cerdanyola
19. Prat
20. Europa


In [97]:
# agrupamos todas las funciones en una función principal que nos devuelva un DataFrame

def main_partidos(lista_partidos):

    partidos_procesados = procesar_partidos(lista_partidos)

    partidos_procesados = normalizar_resultados(partidos_procesados)

    estadisticas_equipos = equipos_participantes(partidos_procesados)

    estadisticas_equipos = inicializar_metricas(estadisticas_equipos)

    estadisticas_equipos = contabilizar_resultados(estadisticas_equipos, partidos_procesados)

    df_partidos = convertir_dataframe(estadisticas_equipos)

    mostrar_resumen(df_partidos)

    return df_partidos

In [98]:
df_partidos = main_partidos(lista_partidos)

Equipo más goleador: Figueres
Equipo más goleado: Vilafranca

Clasificación final:
1. Girona FC
2. Llagostera
3. Sabadell
4. Cornellà
5. RCD Espanyol
6. Figueres
7. Lleida Esportiu
8. Terrassa
9. FC Barcelona
10. Vilafranca
11. Badalona
12. Nàstic de Tarragona
13. Reus Deportiu
14. Granollers
15. Olot
16. Sant Andreu
17. Manlleu
18. Cerdanyola
19. Prat
20. Europa
